In [1]:
# Step 1. Import libraries
import pandas as pd
import os

MIMIC_PATH = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/builtdata/csv_concepts_exports"
MIMIC_CORE_PATH = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/builtdata/csv_exports"

OUTPUT_PATH = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files"

In [2]:
# Step 2. Load GCS data and ICU stays
mimic_gcs_file = os.path.join(MIMIC_PATH, "gcs.csv")
icu_stays_file = os.path.join(MIMIC_CORE_PATH, "icu_icustays.csv")

mimic_gcs = pd.read_csv(mimic_gcs_file)
icu_stays = pd.read_csv(icu_stays_file)

print("MIMIC GCS preview:")
print(mimic_gcs.head())

print("ICU stays preview:")
print(icu_stays.head())

MIMIC GCS preview:
   subject_id   stay_id            charttime  gcs  gcs_motor  gcs_verbal  \
0    12466550  30000153  2174-09-29 12:45:00   15        5.0         0.0   
1    12466550  30000153  2174-09-29 16:26:00   15        6.0         0.0   
2    12466550  30000153  2174-09-29 17:37:00   15        6.0         0.0   
3    12466550  30000153  2174-09-29 18:00:00    9        5.0         1.0   
4    12466550  30000153  2174-09-29 19:00:00    9        5.0         1.0   

   gcs_eyes  gcs_unable  
0       3.0           1  
1       4.0           1  
2       4.0           1  
3       3.0           0  
4       3.0           0  
ICU stays preview:
   subject_id   hadm_id   stay_id                       first_careunit  \
0    10000032  29079034  39553978   Medical Intensive Care Unit (MICU)   
1    10000690  25860671  37081114   Medical Intensive Care Unit (MICU)   
2    10000980  26913865  39765666   Medical Intensive Care Unit (MICU)   
3    10001217  24597018  37067082  Surgical Intensive

In [3]:
# Step 3. Join with icu_stays to bring in hadm_id (as csn)
mimic_gcs_merged = mimic_gcs.merge(
    icu_stays[["subject_id", "hadm_id", "stay_id"]],
    on=["stay_id", "subject_id"],
    how="left"
)

# Step 4. Rename columns to match target schema
mimic_gcs_final = mimic_gcs_merged.rename(columns={
    "subject_id": "pat_id",
    "hadm_id": "csn",
    "charttime": "recorded_time",
    "gcs_eyes": "gcs_eye_score",
    "gcs_verbal": "gcs_verbal_score",
    "gcs_motor": "gcs_motor_score",
    "gcs": "gcs_total_score"
})

# Step 5. Select only needed columns (keep gcs_unable as is)
mimic_gcs_final = mimic_gcs_final[
    ["pat_id", "csn", "recorded_time", 
     "gcs_eye_score", "gcs_verbal_score", "gcs_motor_score", "gcs_total_score", 
     "gcs_unable"]
]

print("Processed MIMIC GCS preview:")
print(mimic_gcs_final.head())

Processed MIMIC GCS preview:
     pat_id       csn        recorded_time  gcs_eye_score  gcs_verbal_score  \
0  12466550  23998182  2174-09-29 12:45:00            3.0               0.0   
1  12466550  23998182  2174-09-29 16:26:00            4.0               0.0   
2  12466550  23998182  2174-09-29 17:37:00            4.0               0.0   
3  12466550  23998182  2174-09-29 18:00:00            3.0               1.0   
4  12466550  23998182  2174-09-29 19:00:00            3.0               1.0   

   gcs_motor_score  gcs_total_score  gcs_unable  
0              5.0               15           1  
1              6.0               15           1  
2              6.0               15           1  
3              5.0                9           0  
4              5.0                9           0  


In [5]:
# Step 6. Export final flat file
outfile = os.path.join(OUTPUT_PATH, "GCS.csv")
mimic_gcs_final.to_csv(outfile, index=False)

print(f"✅ Exported MIMIC GCS flat file to {outfile}")

✅ Exported MIMIC GCS flat file to /hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files/GCS.csv
